# Módulo 2 (continuação) — Rotulagem dos fluxos NFStream a partir dos PCAPs da CICIDS2017
**Objetivo:** reconstruir a coluna `Label` para os fluxos extraídos com NFStream a partir dos PCAPs originais da CICIDS2017, usando IP de origem/destino e o timestamp `bidirectional_first_seen_ms`, com base nas informações publicadas em https://www.unb.ca/cic/datasets/ids-2017.html.
**Autor:** Marcel Teixeira Chiarelo
**Data:** julho/2026


## 1. Contexto

Foi pedido que a rotulagem seja feita **sem** depender do CSV original da CICIDS2017 (gerado pelo CICFlowMeter): a ideia é usar apenas as informações públicas do site do CIC — IP de atacante, IP de vítima e o intervalo de horário de cada ataque — combinadas com `src_ip`, `dst_ip` e `bidirectional_first_seen_ms` (timestamp de início do fluxo) de cada fluxo extraído pelo NFStream.

As janelas de ataque (data, hora de início, hora de fim, atacante, vítima) usadas aqui foram copiadas da própria página do CIC e estão centralizadas no arquivo `rotulagem_cicids2017.py`, junto desta notebook.

**Atenção ao NAT do firewall:** o site do CIC documenta que o tráfego do atacante (Kali, IP público `205.174.165.73`) passa por NAT antes de chegar na rede da vítima: `205.174.165.73 → 205.174.165.80 → 172.16.0.1 → vítima`. Como o PCAP `Thursday-WorkingHours.pcap` foi capturado dentro da rede interna (todos os IPs no seu CSV são da faixa `192.168.10.x`), **o atacante aparece como `172.16.0.1`, não como o IP público**. O módulo `rotulagem_cicids2017.py` já considera os dois IPs como candidatos.

**Limitação conhecida:** a literatura (Engelen, Rimmer & Joosen, 2021 — *Troubleshooting an Intrusion Detection Dataset: the CICIDS2017 Case Study*) aponta que as janelas de horário oficiais não são perfeitamente exatas — os próprios autores do CICFlowMeter precisaram ajustá-las manualmente. Por isso, a Seção 4 desta notebook faz uma checagem comparando a distribuição de classes obtida aqui com a distribuição do CSV original, para termos ideia da qualidade da rotulagem.


In [1]:
import glob
import pandas as pd
from rotulagem_cicids2017 import label_dataframe, ATTACK_WINDOWS, ler_csv_original_cicids2017, normalizar_label

pd.set_option("display.max_columns", None)


## 2. Carregar os CSVs gerados pelo NFStream

Parte do que já foi gerado em `modulo2_nfstream_pcap_cicids2017.ipynb`: o NFStream rodou sobre o `Thursday-WorkingHours.pcap` inteiro e salvou `captura_nfstream_thursday_webattacks.csv` (nesta mesma pasta `modulo2/`). Se depois você processar outros dias (Tuesday/Wednesday/Friday), basta salvar os CSVs seguindo o mesmo padrão de nome e o `glob` abaixo já pega todos.


In [2]:
# Ajuste o padrão de busca se salvar CSVs de outros dias depois
NFSTREAM_CSVS = sorted(glob.glob("captura_nfstream_*.csv"))
print(f"{len(NFSTREAM_CSVS)} arquivo(s) encontrado(s):")
for f in NFSTREAM_CSVS:
    print(" -", f)


1 arquivo(s) encontrado(s):
 - captura_nfstream_thursday_webattacks.csv


In [3]:
dfs = []
for f in NFSTREAM_CSVS:
    d = pd.read_csv(f)
    d["__origem_arquivo"] = f
    dfs.append(d)

df_nfstream = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"Total de fluxos carregados: {len(df_nfstream)}")
df_nfstream.head()


Total de fluxos carregados: 380135


,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,protocol,ip_version,vlan_id,tunnel_id,bidirectional_first_seen_ms,bidirectional_last_seen_ms,bidirectional_duration_ms,bidirectional_packets,bidirectional_bytes,src2dst_first_seen_ms,src2dst_last_seen_ms,src2dst_duration_ms,src2dst_packets,src2dst_bytes,dst2src_first_seen_ms,dst2src_last_seen_ms,dst2src_duration_ms,dst2src_packets,dst2src_bytes,bidirectional_min_ps,bidirectional_mean_ps,bidirectional_stddev_ps,bidirectional_max_ps,src2dst_min_ps,src2dst_mean_ps,src2dst_stddev_ps,src2dst_max_ps,dst2src_min_ps,dst2src_mean_ps,dst2src_stddev_ps,dst2src_max_ps,bidirectional_min_piat_ms,bidirectional_mean_piat_ms,bidirectional_stddev_piat_ms,bidirectional_max_piat_ms,src2dst_min_piat_ms,src2dst_mean_piat_ms,src2dst_stddev_piat_ms,src2dst_max_piat_ms,dst2src_min_piat_ms,dst2src_mean_piat_ms,dst2src_stddev_piat_ms,dst2src_max_piat_ms,bidirectional_syn_packets,bidirectional_cwr_packets,bidirectional_ece_packets,bidirectional_urg_packets,bidirectional_ack_packets,bidirectional_psh_packets,bidirectional_rst_packets,bidirectional_fin_packets,src2dst_syn_packets,src2dst_cwr_packets,src2dst_ece_packets,src2dst_urg_packets,src2dst_ack_packets,src2dst_psh_packets,src2dst_rst_packets,src2dst_fin_packets,dst2src_syn_packets,dst2src_cwr_packets,dst2src_ece_packets,dst2src_urg_packets,dst2src_ack_packets,dst2src_psh_packets,dst2src_rst_packets,dst2src_fin_packets,application_name,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type,__origem_arquivo
0,0,0,192.168.10.50,00:19:b9:0a:69:f1,00:19:b9,33898,192.168.10.3,18:66:da:9b:e3:7d,18:66:da,389,6,4,0,0,1499342340490,1499342340490,0,6,1834,1499342340490,1499342340490,0,4,1070,1499342340490,1499342340490,0,2,764,66,305.666667,189.678324,469,66,267.5,232.672158,469,382,382.0,0.0,382,0,0.000000,0.00000,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0,0,0,6,4,0,0,0,0,0,0,4,2,0,0,0,0,0,0,2,2,0,0,LDAP,System,1,1,NaN,NaN,NaN,NaN,NaN,captura_nfstream_thursday_webattacks.csv
1,1,0,192.168.10.50,00:19:b9:0a:69:f1,00:19:b9,33904,192.168.10.3,18:66:da:9b:e3:7d,18:66:da,389,6,4,0,0,1499342340491,1499342340491,0,6,1834,1499342340491,1499342340491,0,4,1070,1499342340491,1499342340491,0,2,764,66,305.666667,189.678324,469,66,267.5,232.672158,469,382,382.0,0.0,382,0,0.000000,0.00000,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0,0,0,6,4,0,0,0,0,0,0,4,2,0,0,0,0,0,0,2,2,0,0,LDAP,System,1,1,NaN,NaN,NaN,NaN,NaN,captura_nfstream_thursday_webattacks.csv
2,2,0,192.168.10.50,00:19:b9:0a:69:f1,00:19:b9,33898,192.168.10.3,18:66:da:9b:e3:7d,18:66:da,389,6,4,0,0,1499342356973,1499342356973,0,6,1834,1499342356973,1499342356973,0,4,1070,1499342356973,1499342356973,0,2,764,66,305.666667,189.678324,469,66,267.5,232.672158,469,382,382.0,0.0,382,0,0.000000,0.00000,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0,0,0,6,4,0,0,0,0,0,0,4,2,0,0,0,0,0,0,2,2,0,0,LDAP,System,1,1,NaN,NaN,NaN,NaN,NaN,captura_nfstream_thursday_webattacks.csv
3,3,0,192.168.10.50,00:19:b9:0a:69:f1,00:19:b9,33904,192.168.10.3,18:66:da:9b:e3:7d,18:66:da,389,6,4,0,0,1499342356974,1499342356974,0,6,1834,1499342356974,1499342356974,0,4,1070,1499342356974,1499342356974,0,2,764,66,305.666667,189.678324,469,66,267.5,232.672158,469,382,382.0,0.0,382,0,0.000000,0.00000,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0,0,0,6,4,0,0,0,0,0,0,4,2,0,0,0,0,0,0,2,2,0,0,LDAP,System,1,1,NaN,NaN,NaN,NaN,NaN,captura_nfstream_thursday_webattacks.csv
4,4,0,192.168.10.14,b8:ac:6f:36:07:ee,b8:ac:6f,59555,192.168.10.3,18:66:da:9b:e3:7d,18:66:da,53,17,4,0,0,1499342357298,1499342357299,1,4,592,1499342357298,1499342357298,0,2,186,1499342357299,1499342357299,0,2,406,93,148.000000,63.508530,203,93,93.0,0.000000,93,203,203.0,0.0,203,0,0.333333,0.57735,1,0,0.0,0.0,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,DNS.Microsoft,Network,0,6,v10.vortex-win.data.microsoft.com,NaN,NaN,NaN,NaN,captura_nfstream_thursday_webattacks.csv


## 3. Aplicar a rotulagem

Cada fluxo é comparado contra as janelas de ataque (`ATTACK_WINDOWS`, em `rotulagem_cicids2017.py`): se `src_ip`/`dst_ip` corresponderem ao par atacante/vítima da janela (considerando o IP pós-NAT `172.16.0.1`) **e** `bidirectional_first_seen_ms` cair dentro do intervalo de horário, o fluxo recebe o rótulo do ataque; caso contrário, recebe `BENIGN`.


In [4]:
df_rotulado = label_dataframe(
    df_nfstream,
    src_col="src_ip",
    dst_col="dst_ip",
    time_col="bidirectional_first_seen_ms",
)

df_rotulado["Label"].value_counts()


Label
BENIGN                        322760
Infiltration                   55356
Web Attack - Brute Force        1349
Web Attack - XSS                 661
Web Attack - Sql Injection         9
Name: count, dtype: int64

In [5]:
df_rotulado.to_csv("nfstream_cicids2017_rotulado.csv", index=False)
print("Arquivo salvo: nfstream_cicids2017_rotulado.csv")
print(f"Dimensões: {df_rotulado.shape}")


Arquivo salvo: nfstream_cicids2017_rotulado.csv
Dimensões: (380135, 88)


## 4. Checagem de sanidade (comparação rápida com o CSV original)

Antes de seguir para a análise estatística completa (próxima notebook), vale comparar rapidamente a contagem de classes obtida aqui com a contagem dos CSVs originais da CICIDS2017 para quinta-feira. Como o NFStream processou o **dia inteiro**, concatenamos os dois CSVs oficiais de quinta (`Morning-WebAttacks` + `Afternoon-Infilteration`) para a comparação ficar justa (dia completo dos dois lados). Diferenças de alguns % são esperadas (ver limitação na Seção 1); diferenças muito grandes (ex.: 10x) indicam problema de fuso horário ou de par de IPs.

Observação: o CSV original que vocês baixaram é a versão "reduzida" da CICIDS2017 (sem colunas de IP/porta/timestamp), então essa checagem compara apenas a **contagem por classe**, não faz cruzamento fluxo a fluxo.


In [6]:
ORIGINAL_CSVS = sorted(glob.glob("dados/Thursday-*.csv"))
print("CSVs originais encontrados:", ORIGINAL_CSVS)

if ORIGINAL_CSVS:
    # BUG #1 (corrigido): antes liamos com encoding="latin1" fixo, o que
    # decodificava silenciosamente um CSV UTF-8 de forma errada e corrompia
    # os rotulos com en-dash. ler_csv_original_cicids2017() detecta o
    # encoding certo automaticamente.
    orig = ler_csv_original_cicids2017(ORIGINAL_CSVS)
    print("Rotulos unicos no CSV original:", sorted(orig["Label"].unique()))

    # BUG #1b: mesmo com o encoding certo, alguns rotulos do CSV original ja
    # chegam com o caractere de travessao substituido por U+FFFD (perdido
    # antes do arquivo chegar ate nos -- ver Secao 4 da notebook
    # modulo2_analise_comparativa.ipynb para detalhes). Por isso comparamos
    # por uma chave normalizada (normalizar_label), nao pela string exata.
    contagem_nosso = df_rotulado["Label"].value_counts()
    contagem_orig = orig["Label"].value_counts()

    norm_para_nome = {}
    for label in list(contagem_nosso.index) + list(contagem_orig.index):
        norm_para_nome.setdefault(normalizar_label(label), label)

    comparacao = pd.DataFrame({
        "NFStream (nosso)": contagem_nosso.groupby(normalizar_label).sum(),
        "CICIDS2017 (original, dia inteiro)": contagem_orig.groupby(normalizar_label).sum(),
    }).fillna(0).astype(int)
    comparacao.index = comparacao.index.map(norm_para_nome)
    comparacao.index.name = "Label"
    display(comparacao)
else:
    print("Nenhum CSV original encontrado em dados/Thursday-*.csv \u2014 ajuste o caminho para rodar a checagem.")


CSVs originais encontrados: ['dados/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'dados/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv']
[ler_csv_original_cicids2017] dados/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: lido com encoding='utf-8'
[ler_csv_original_cicids2017] dados/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: lido com encoding='utf-8'
Rotulos unicos no CSV original: ['BENIGN', 'Infiltration', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']


,NFStream (nosso),"CICIDS2017 (original, dia inteiro)"
Label,,
BENIGN,322760,456752
Infiltration,55356,36
Web Attack - Brute Force,1349,1507
Web Attack - Sql Injection,9,21
Web Attack - XSS,661,652


## Reflexão

- **Fluxos rotulados:** dos 380.135 fluxos gerados pelo NFStream para quinta-feira,
  322.760 (84,9%) ficaram como BENIGN e 57.375 (15,1%) foram rotulados como algum
  tipo de ataque — sendo 55.356 Infiltration, 1.349 Web Attack - Brute Force,
  661 Web Attack - XSS e 9 Web Attack - Sql Injection.

- **A distribuição ficou longe da distribuição do CSV original** (que tem 99,52%
  BENIGN e 0,48% ataque). A rotulagem por IP+tempo classifica bem mais tráfego
  como ataque do que o CICFlowMeter/CIC original.

- **Divergência por classe:**
  - *Web Attack (Brute Force, XSS, Sql Injection):* ficaram próximas do CSV
    original (1.349 vs 1.507, 661 vs 652, 9 vs 21) — divergências de poucos por
    cento, dentro do esperado pela imprecisão conhecida das janelas de horário
    publicadas pelo CIC (ver Seção 1, Engelen, Rimmer & Joosen, 2021). Essas
    janelas usam pares de IP específicos (atacante externo + IP do NAT), então
    o casamento é preciso.
  - *Infiltration:* divergência grande (55.356 vs 36 no CSV original). Esse
    ataque tem uma segunda etapa em que a própria máquina infectada
    (`192.168.10.8`) faz um portscan interno contra a rede — como não há um par
    específico de atacante/vítima documentado pelo CIC para essa etapa, a janela
    usa um critério mais amplo (qualquer tráfego de/para `192.168.10.8` que não
    seja com servidores de infraestrutura conhecidos), o que ainda captura mais
    tráfego do que só o scan propriamente dito. Isso é agravado pela forma como
    o NFStream constrói fluxos (sem o timeout de 120s do CICFlowMeter), o que
    tende a fragmentar o mesmo tráfego bruto em muito mais fluxos individuais.
    A literatura (Engelen et al., 2021) já aponta a classe Infiltration
    especificamente como uma das mais problemáticas em termos de rotulagem no
    CSV oficial da CICIDS2017 — então parte dessa divergência não é um erro da
    nossa rotulagem, e sim uma limitação conhecida do próprio dataset de
    referência.

- **Limitação de comparação identificada nesta rodada:** o CSV original tem
  alguns rótulos com o caractere de travessão corrompido (aparece como
  caractere de substituição, ex. "Web Attack � XSS"), então a tabela de
  comparação acima mostra 0 nessas linhas do lado "original" — os valores
  reais (1.507/21/652) aparecem no print de rótulos únicos, logo acima da
  tabela. Fica registrado como limitação a corrigir numa próxima rodada.

- Este arquivo (`nfstream_cicids2017_rotulado.csv`) é a entrada da notebook
  `modulo2_analise_comparativa.ipynb`.
